In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **1. Import Lib**


In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, f1_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import joblib


# **2. Load cleaned data**


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/datawarehouse/data_cleaned.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      36457 non-null  int64  
 1   CODE_GENDER             36457 non-null  object 
 2   FLAG_OWN_CAR            36457 non-null  object 
 3   FLAG_OWN_REALTY         36457 non-null  object 
 4   AMT_INCOME_TOTAL        36457 non-null  float64
 5   NAME_INCOME_TYPE        36457 non-null  object 
 6   NAME_EDUCATION_TYPE     36457 non-null  object 
 7   NAME_FAMILY_STATUS      36457 non-null  object 
 8   NAME_HOUSING_TYPE       36457 non-null  object 
 9   FLAG_WORK_PHONE         36457 non-null  int64  
 10  FLAG_PHONE              36457 non-null  int64  
 11  FLAG_EMAIL              36457 non-null  int64  
 12  OCCUPATION_TYPE         36457 non-null  object 
 13  TARGET                  36457 non-null  int64  
 14  AGE                     36457 non-null

# **3. Train test split**


In [ ]:
X = df.drop(columns=["TARGET", "ID"], axis=1)
y = df["TARGET"]

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CODE_GENDER             36457 non-null  object 
 1   FLAG_OWN_CAR            36457 non-null  object 
 2   FLAG_OWN_REALTY         36457 non-null  object 
 3   AMT_INCOME_TOTAL        36457 non-null  float64
 4   NAME_INCOME_TYPE        36457 non-null  object 
 5   NAME_EDUCATION_TYPE     36457 non-null  object 
 6   NAME_FAMILY_STATUS      36457 non-null  object 
 7   NAME_HOUSING_TYPE       36457 non-null  object 
 8   FLAG_WORK_PHONE         36457 non-null  int64  
 9   FLAG_PHONE              36457 non-null  int64  
 10  FLAG_EMAIL              36457 non-null  int64  
 11  OCCUPATION_TYPE         36457 non-null  object 
 12  AGE                     36457 non-null  int64  
 13  YEARS_EMPLOYED          36457 non-null  int64  
 14  CNT_CHILDREN_BINNED     36457 non-null

In [ ]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 36457 entries, 0 to 36456
Series name: TARGET
Non-Null Count  Dtype
--------------  -----
36457 non-null  int64
dtypes: int64(1)
memory usage: 284.9 KB


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **4. Oversampling train_data**


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# # Encode categorical variables
# categorical_cols = X.select_dtypes(include=['object']).columns

# for col in categorical_cols:
#     one_hot = pd.get_dummies(X_train[col], prefix=col)
#     X_train = X_train.drop(col, axis=1)
#     X_train = pd.concat([X_train, one_hot], axis=1)


# smote = SMOTE(random_state=42)
# X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
# print("Before SMOTE:", y_train.value_counts())
# print("After SMOTE:", y_train_res.value_counts())
# print("Before SMOTE shape:", X_train.shape)
# print("After SMOTE shape:", X_train_res.shape)

# **5. Create Preprocessor**


In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
category_cols = X_train.select_dtypes(include="object").columns.tolist()

In [ ]:
# Numeric pipeline
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Category pipline
category_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine pipelines into Workflow
preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", category_transformer, category_cols)
])

# **6. Setup models**


In [ ]:
# models = {
#     "DecisionTree": Pipeline([
#         ("prep", preprocess),
#         ("clf", DecisionTreeClassifier(
#             random_state=42,
#             class_weight="balanced"
#         ))
#     ]),

#     "RandomForest": Pipeline([
#         ("prep", preprocess),
#         ("clf", RandomForestClassifier(
#             random_state=42,
#             class_weight="balanced_subsample"
#         ))
#     ]),

#     "GradientBoosting": Pipeline([
#         ("prep", preprocess),
#         ("clf", GradientBoostingClassifier(
#             random_state=42
#         ))
#     ]),

#     "AdaBoost": Pipeline([
#         ("prep", preprocess),
#         ("clf", AdaBoostClassifier(
#             random_state=42
#         ))
#     ]),

#     "XGBoost": Pipeline([
#         ("prep", preprocess),
#         ("clf", XGBClassifier(
#             random_state=42,
#             use_label_encoder=False,
#             eval_metric='logloss'
#         ))
#     ]),

#     "LightGBM": Pipeline([
#         ("prep", preprocess),
#         ("clf", LGBMClassifier(
#             random_state=42
#         ))
#     ]),

#     "LogisticRegression": Pipeline([
#         ("prep", preprocess),
#         ("clf", LogisticRegression(
#             random_state=42,
#             max_iter=1000,
#             class_weight="balanced"
#         ))
#     ]),

#     "SVC": Pipeline([
#         ("prep", preprocess),
#         ("clf", SVC(
#             random_state=42,
#             class_weight="balanced",
#             probability=True
#         ))
#     ])
# }

In [ ]:
models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(random_state=42, class_weight="balanced_subsample"),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
}

In [ ]:
# Params for GridSearchCV
params = {
    "DecisionTree": {
        "clf__max_depth": [None, 10, 20, 30],
        "clf__min_samples_split": [2, 5, 10],
        "clf__min_samples_leaf": [1, 2, 4]
    },

    "RandomForest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5],
        "clf__min_samples_leaf": [1, 2]
    },

    "GradientBoosting": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.01, 0.1, 0.2],
        "clf__max_depth": [3, 5, 7]
    },

    "AdaBoost": {
        "clf__n_estimators": [50, 100, 200],
        "clf__learning_rate": [0.01, 0.1, 0.2]
    },

    "XGBoost": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.01, 0.1, 0.2],
        "clf__max_depth": [3, 5, 7]
    },

    "LightGBM": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.01, 0.1, 0.2],
        "clf__num_leaves": [31, 50, 100]
    },

    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1, 10, 100],
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "saga"]
    }
}

# **7. Run Cross Validation**


In [ ]:
cv_result = {}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"Running CV for {name}...")

    pipe = ImbPipeline([
        ("prep", preprocess),
        ("smote", SMOTE(random_state=42)),
        ("clf", model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="average_precision", n_jobs=-1)

    cv_result[name] = {
        "average_precision_mean": np.mean(scores),
        "average_precision_std": np.std(scores)
    }

print(f"Results from Cross Validation:")
for name, result in cv_result.items():
    print(f"{name}: average_precision = {result['average_precision_mean']:.4f} +- {result['average_precision_std']:.4f}")

Running CV for DecisionTree...
Running CV for RandomForest...
Running CV for GradientBoosting...
Running CV for AdaBoost...
Running CV for XGBoost...
Running CV for LightGBM...
Running CV for LogisticRegression...
Results from Cross Validation:
DecisionTree: average_precision = 0.1200 +- 0.0165
RandomForest: average_precision = 0.1985 +- 0.0239
GradientBoosting: average_precision = 0.0405 +- 0.0088
AdaBoost: average_precision = 0.0233 +- 0.0048
XGBoost: average_precision = 0.1689 +- 0.0201
LightGBM: average_precision = 0.1269 +- 0.0222
LogisticRegression: average_precision = 0.0359 +- 0.0149


In [ ]:
# gg

In [ ]:
grid_result = {}

for name, model in models.items():
    print(f"Running GridSearchCV for {name}...")
    param_grid = params.get(name, {})
    pipe = ImbPipeline([
        ("prep", preprocess),
        ("smote", SMOTE(random_state=42)),
        ("clf", model)
    ])
    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="average_precision",
        cv=kf,
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)
    grid_result[name] = grid_search
    print(f"Best parameters for {name}: {grid_search.best_params_}")
    print(f"Best average_precision for {name}: {grid_search.best_score_:.4f}")

Running GridSearchCV for DecisionTree...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters for DecisionTree: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Best ROC_AUC for DecisionTree: 0.1200
Running GridSearchCV for RandomForest...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters for RandomForest: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}
Best ROC_AUC for RandomForest: 0.2012
Running GridSearchCV for GradientBoosting...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best parameters for GradientBoosting: {'clf__learning_rate': 0.2, 'clf__max_depth': 7, 'clf__n_estimators': 200}
Best ROC_AUC for GradientBoosting: 0.2177
Running GridSearchCV for AdaBoost...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best parameters for AdaBoost: {'clf__learning_rate': 0.2, 'clf__n_estimators': 200}
Best ROC_AUC for AdaBoost: 0.0241
Running GridSearchCV for XGBoost...
Fitting 5 folds for each of 18 candidates, totalling 90 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:56:10] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters for XGBoost: {'clf__learning_rate': 0.2, 'clf__max_depth': 7, 'clf__n_estimators': 200}
Best ROC_AUC for XGBoost: 0.1982
Running GridSearchCV for LightGBM...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
[LightGBM] [Info] Number of positive: 28672, number of negative: 28672
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018659 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10219
[LightGBM] [Info] Number of data points in the train set: 57344, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Best parameters for LightGBM: {'clf__learning_rate': 0.2, 'clf__n_estimators': 200, 'clf__num_leaves': 50}
Best ROC_AUC for LightGBM: 0.2209
Running GridSearchCV for LogisticRegression...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best paramet

# **8. Set up Best_models**

In [ ]:
best_models = {
    "DecisionTree": grid_result["DecisionTree"].best_estimator_,
    "RandomForest": grid_result["RandomForest"].best_estimator_,
    "GradientBoosting": grid_result["GradientBoosting"].best_estimator_,
    "AdaBoost": grid_result["AdaBoost"].best_estimator_,
    "XGBoost": grid_result["XGBoost"].best_estimator_,
    "LightGBM": grid_result["LightGBM"].best_estimator_,
    "LogisticRegression": grid_result["LogisticRegression"].best_estimator_
}

# **9. Predict on test dataset**

In [ ]:
def evaluate_classifier(y_true, y_pred, y_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba[:, 1]) if y_proba is not None else None

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    }


In [ ]:
rows, artifacts = [], {}
threshold = 0.25
for name, pipe in best_models.items():
  y_pred = pipe.predict(X_test)
  y_pred_modified = (y_pred >= threshold).astype(inSt)
  y_proba = pipe.predict_proba(X_test)
  metrics = evaluate_classifier(y_pred=y_pred_modified, y_true=y_test, y_proba=y_proba)
  rows.append({"Models": name, "Threshold": threshold, **metrics})

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
metrics_df = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False)
metrics_df

,Models,Threshold,Accuracy,Precision,Recall,F1 Score,ROC AUC
1,RandomForest,0.25,0.979430,0.333333,0.219512,0.264706,0.797634
4,XGBoost,0.25,0.979018,0.317073,0.211382,0.253659,0.731862
5,LightGBM,0.25,0.979841,0.357143,0.243902,0.289855,0.721015
2,GradientBoosting,0.25,0.979704,0.352941,0.243902,0.288462,0.718154
0,DecisionTree,0.25,0.976413,0.271028,0.235772,0.252174,0.698982
3,AdaBoost,0.25,0.531130,0.019814,0.552846,0.038256,0.561945
6,LogisticRegression,0.25,0.553620,0.019337,0.512195,0.037267,0.556220


# **10. Save models**

In [ ]:
# Save models
for name, grid_search in grid_result.items():
    model_filename = f"/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/{name}_model.pkl"
    joblib.dump(grid_search.best_estimator_, model_filename)
    print(f"Saved {name} model to {model_filename}")

Saved DecisionTree model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/DecisionTree_model.pkl
Saved RandomForest model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/RandomForest_model.pkl
Saved GradientBoosting model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/GradientBoosting_model.pkl
Saved AdaBoost model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/AdaBoost_model.pkl
Saved XGBoost model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/XGBoost_model.pkl
Saved LightGBM model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/LightGBM_model.pkl
Saved LogisticRegression model to /content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/LogisticRegression_model.pkl


# **11. Features impact on Target**

In [ ]:
best_model = joblib.load("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/models/RandomForest_model.pkl")

prep = best_model.named_steps["prep"]
clf = best_model.named_steps["clf"]
importances = clf.feature_importances_

num_cols = prep.transformers_[0][2]
ohe = prep.named_transformers_["cat"].named_steps["one_hot"]
cat_cols = prep.transformers_[1][2]
cat_names = list(ohe.get_feature_names_out(cat_cols))

feat_names = list(num_cols) + cat_names

fi_df = pd.DataFrame({
    "feature_transformed": feat_names,
    "importance": importances
}).sort_values("importance", ascending=False)
fi_df.head(10)


,feature_transformed,importance
4,AGE,0.105545
0,AMT_INCOME_TOTAL,0.101358
5,YEARS_EMPLOYED,0.071871
10,FLAG_OWN_REALTY_N,0.042423
11,FLAG_OWN_REALTY_Y,0.036970
28,OCCUPATION_TYPE_General Laborers,0.036840
29,OCCUPATION_TYPE_High-Level Professional,0.033387
30,OCCUPATION_TYPE_Service & Sales,0.033178
15,NAME_INCOME_TYPE_Working,0.031097
21,NAME_FAMILY_STATUS_Married,0.030876


In [ ]:
fi_df.to_csv("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/datawarehouse/feature_importance.csv", index=False)
print("Saved feature importance to feature_importance.csv")

Saved feature importance to feature_importance.csv


# **12. Run what if analysis**

In [ ]:
# Define scenario
WHAT_IFS = [
    ("At least 3 years of work experience", {"YEARS_EMPLOYED": (3, ">=")}),
    ("At least 29 years old", {"AGE": (29, ">=")}),
    ("Minimum income of 150,000", {"AMT_INCOME_TOTAL": (150000, ">=")}),
    ("Minimum income of 90,000", {"AMT_INCOME_TOTAL": (90000, ">=")}),
    ("Real estate ownership required", {"FLAG_OWN_REALTY": ("Y", "==")}),
    ("Car ownership required", {"FLAG_OWN_CAR": ("Y", "==")}),
    ("Do not accept pensioners", {"NAME_INCOME_TYPE": ("Pensioner", "!=")}),
    ("Do not accept widows/widowers", {"NAME_FAMILY_STATUS": ("Widow", "!=")}),
    ("Do not accept single / not married", {"NAME_FAMILY_STATUS": ("Single / not married", "!=")}),
    ("Do not accept high-level professionals", {"OCCUPATION_TYPE": ("High-Level Professional", "!=")}),
    ("Do not accept applicants with lower secondary education", {"NAME_EDUCATION_TYPE": ("Lower secondary", "!=")}),
    ("Do not accept applicants with Incomplete higher education", {"NAME_EDUCATION_TYPE": ("Incomplete higher", "!=")}),
]

In [ ]:
def evaluate_scenario(X_df, y_df, model, criteria_dict, threshold=0.25):
    def check_criteria(row):
        for col, (val, op) in criteria_dict.items():
            if op == ">=":
                if not (row[col] >= val): return False
            elif op == "==":
                if not (row[col] == val): return False
            elif op == "!=":
                if not (row[col] != val): return False
        return True

    # Filter accepted data
    passed_mask = X_df.apply(check_criteria, axis=1)
    X_accepted = X_df[passed_mask]
    y_actual_accepted = y_df[passed_mask]

    if len(X_accepted) == 0:
        return 0, 0

    # Compute bad rate for accepted data
    actual_bad_rate = (y_actual_accepted == 1).mean() * 100

    # Compute
    rej_count = len(X_df) - len(X_accepted)

    return actual_bad_rate, rej_count

In [ ]:
results = []
# Compute baseline_bad_rate to compare
baseline_rate = (y_test == 1).mean() * 100
baseline_count = len(X_test)

for title, criteria in WHAT_IFS:
    bad_rate, rej_count = evaluate_scenario(X_test, y_test, best_model, criteria)

    results.append({
        "Scenario": title,
        "Bad Debt Rate (%)": round(bad_rate, 3),
        "Difference Bad Rate (%)": round(bad_rate - baseline_rate, 3),
        "Rejected Count": rej_count,
        "Reject Rate (%)": round(rej_count / baseline_count * 100, 3)
    })

df_what_if = pd.DataFrame(results).sort_values("Difference Bad Rate (%)", ascending=True)

df_what_if

,Scenario,Bad Debt Rate (%),Difference Bad Rate (%),Rejected Count,Reject Rate (%)
0,At least 3 years of work experience,1.455,-0.232,2962,40.620
4,Real estate ownership required,1.458,-0.229,2353,32.268
2,"Minimum income of 150,000",1.495,-0.192,3078,42.211
9,Do not accept high-level professionals,1.538,-0.149,1895,25.987
5,Car ownership required,1.571,-0.116,4555,62.466
6,Do not accept pensioners,1.572,-0.115,1249,17.128
7,Do not accept widows/widowers,1.612,-0.075,343,4.704
3,"Minimum income of 90,000",1.620,-0.067,562,7.707
8,Do not accept single / not married,1.620,-0.067,934,12.809
11,Do not accept applicants with Incomplete highe...,1.637,-0.049,269,3.689


In [ ]:
df_what_if.to_csv("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/datawarehouse/what_if_analysis.csv", index=False)
print("Saved what if analysis to what_if_analysis.csv")

Saved what if analysis to what_if_analysis.csv


In [ ]:
rows, artifacts = [], {}
threshold = 0.25
name = "RandomForest"
y_pred = best_model.predict(X_test)
y_pred_modified = (y_pred >= threshold).astype(int)
y_proba = best_model.predict_proba(X_test)
metrics = evaluate_classifier(y_pred=y_pred_modified, y_true=y_test, y_proba=y_proba)
rows.append({"Models": name, "Threshold": threshold, **metrics})

In [ ]:
metrics_df = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False)
metrics_df

,Models,Threshold,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,RandomForest,0.25,0.97943,0.333333,0.219512,0.264706,0.797634


In [ ]:
metrics_df.to_csv("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/datawarehouse/metrics.csv", index=False)
print("Saved metrics to metrics.csv")

Saved metrics to metrics.csv


# **13. Create sample data for testing LLM reasoning**

In [ ]:
data_label_1 = df[df["TARGET"]==1].head(30)
data_label_0 = df[df["TARGET"]==0].head(30)

In [ ]:
sample_data = pd.concat([data_label_0, data_label_1], axis=0, ignore_index=True)
sample_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      60 non-null     int64  
 1   CODE_GENDER             60 non-null     object 
 2   FLAG_OWN_CAR            60 non-null     object 
 3   FLAG_OWN_REALTY         60 non-null     object 
 4   AMT_INCOME_TOTAL        60 non-null     float64
 5   NAME_INCOME_TYPE        60 non-null     object 
 6   NAME_EDUCATION_TYPE     60 non-null     object 
 7   NAME_FAMILY_STATUS      60 non-null     object 
 8   NAME_HOUSING_TYPE       60 non-null     object 
 9   FLAG_WORK_PHONE         60 non-null     int64  
 10  FLAG_PHONE              60 non-null     int64  
 11  FLAG_EMAIL              60 non-null     int64  
 12  OCCUPATION_TYPE         60 non-null     object 
 13  TARGET                  60 non-null     int64  
 14  AGE                     60 non-null     int6

In [ ]:
sample_data.to_csv("/content/drive/MyDrive/ADV_02/capstone_cuoi_khoa/datawarehouse/sample_data.csv", index=False)
print("Saved sample data to sample_data.csv")

Saved sample data to sample_data.csv
